## 1) Kaggle Notebook：依赖安装与目录挂载

> 以下单元是“复制到 Kaggle 后”直接运行的模板。
>
> - `/kaggle/input`：只读，放你上传的代码包和数据；
> - `/kaggle/working`：可写，保存训练日志和模型；
> - 如果你已经把 `export_unet` 目录直接上传（非 zip），可跳过解压步骤。

In [ ]:
# Kaggle 环境初始化（在 Kaggle 执行）
import os
import sys
import glob
import random
import numpy as np
import torch

# 可选：如果环境缺少依赖，取消注释安装
# !pip -q install -U pillow matplotlib tqdm

# 1) 固定随机种子，保证结果尽量可复现
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 2) 自动选择设备：优先 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# 3) 代码解压路径（你上传的 Dataset 名字需要自己替换）
# 例如：/kaggle/input/my-unet-code/export_unet.zip
zip_candidate = "/kaggle/input/my-unet-code/export_unet.zip"
work_dir = "/kaggle/working/unet_run"
os.makedirs(work_dir, exist_ok=True)

if os.path.exists(zip_candidate):
    import zipfile
    with zipfile.ZipFile(zip_candidate, "r") as zf:
        zf.extractall(work_dir)
    print("已解压代码到:", work_dir)
else:
    print("未找到 zip，开始自动扫描 /kaggle/input 下的代码目录")

# 4) 自动发现包含 unet/unet_model.py 的目录，并加入 Python 搜索路径
search_roots = [work_dir, "/kaggle/input"]
candidate_roots = []
for root in search_roots:
    if os.path.exists(root):
        hits = glob.glob(os.path.join(root, "**", "unet", "unet_model.py"), recursive=True)
        for h in hits:
            # 如果命中 .../<root>/unet/unet_model.py，则 package root 应该是其上一级目录
            pkg_root = os.path.dirname(os.path.dirname(h))
            candidate_roots.append(pkg_root)

candidate_roots = sorted(set(candidate_roots))
for p in candidate_roots:
    if p not in sys.path:
        sys.path.insert(0, p)

print("已加入的候选路径数量:", len(candidate_roots))
for p in candidate_roots[:10]:
    print(" -", p)

# 5) 立即做一次导入自检，避免后续训练单元才报错
try:
    from unet.unet_model import UNet
    print("UNet 导入成功")
except Exception as e:
    print("UNet 导入失败，请检查你的 Kaggle 数据集是否包含 unet 目录")
    print("错误信息:", e)
    print("建议检查: /kaggle/input/<your-dataset>/.../unet/unet_model.py 是否存在")

## 2) Kaggle Notebook：数据加载与预处理管线

本节给出一个最小可改造的数据管线：

- 输入图像：RGB，输出张量维度 `N,C,H,W`
- 掩码：单通道类别索引，维度 `N,H,W`
- 二分类常见编码：`0` 背景，`1` 前景（若原始是 0/255，会在读取时映射）

In [ ]:
import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

class SegDataset(Dataset):
    def __init__(self, img_paths, mask_paths, image_size=(256, 256)):
        assert len(img_paths) == len(mask_paths), "图像与掩码数量必须一致"
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.image_size = image_size

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # 1) 读取图像并统一到 RGB
        img = Image.open(self.img_paths[idx]).convert("RGB").resize(self.image_size)
        # 2) 读取掩码并统一到 L（灰度）
        mask = Image.open(self.mask_paths[idx]).convert("L").resize(self.image_size)

        # 3) 转 numpy 并做归一化/编码
        img = np.array(img).astype(np.float32) / 255.0  # [H,W,C]
        mask = np.array(mask)

        # 若你的掩码本来是 0/255，则映射成 0/1 类别索引
        mask = (mask > 127).astype(np.int64)

        # 4) 维度转成 PyTorch 习惯：图像 [C,H,W]，mask [H,W]
        img = np.transpose(img, (2, 0, 1))

        return torch.tensor(img, dtype=torch.float32), torch.tensor(mask, dtype=torch.long)

# 你在 Kaggle 里把这里改成你的数据路径
IMG_GLOB = "/kaggle/input/your-data/images/*"
MASK_GLOB = "/kaggle/input/your-data/masks/*"

img_list = sorted(glob.glob(IMG_GLOB))
mask_list = sorted(glob.glob(MASK_GLOB))

print("img_count:", len(img_list), "mask_count:", len(mask_list))

if len(img_list) > 0 and len(mask_list) > 0:
    ds = SegDataset(img_list, mask_list, image_size=(256, 256))
    dl = DataLoader(ds, batch_size=4, shuffle=True, num_workers=2)

    # 可视化检查：确保图像和掩码对齐
    sample_img, sample_mask = ds[0]
    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.title("Image")
    plt.imshow(np.transpose(sample_img.numpy(), (1, 2, 0)))
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.title("Mask")
    plt.imshow(sample_mask.numpy(), cmap="gray")
    plt.axis("off")
    plt.show()
else:
    print("请先替换 IMG_GLOB / MASK_GLOB 为你的 Kaggle 数据路径")

## 3) Kaggle Notebook：UNet 训练与验证（高注释版）

> 说明：本节给最小可运行训练框架，支持：
> - `BCEWithLogitsLoss`（单通道输出）
> - `CrossEntropyLoss`（多类别输出）
> - 混合精度（AMP）
> - 学习率调度
> - 最优权重保存

In [ ]:
from tqdm import tqdm

# 导入你抽离出来的 UNet
# 这里做一层兜底：若直接导入失败，自动再扫描一次路径
try:
    from unet.unet_model import UNet
except ModuleNotFoundError:
    import os
    import sys
    import glob

    fallback_hits = glob.glob("/kaggle/input/**/unet/unet_model.py", recursive=True)
    for h in fallback_hits:
        pkg_root = os.path.dirname(os.path.dirname(h))
        if pkg_root not in sys.path:
            sys.path.insert(0, pkg_root)

    from unet.unet_model import UNet

# 关键超参数（可改）
NUM_CLASSES = 2           # 二分类可设为2（CE）或1（BCE）
EPOCHS = 5
BATCH_SIZE = 4
LR = 1e-4
USE_AMP = True

# 如果上节未成功构建 dl，这里先兜底
if "dl" not in globals() or dl is None:
    print("未检测到 DataLoader，请先完成第9节的数据路径配置")
else:
    model = UNet(n_channels=3, n_classes=NUM_CLASSES, bilinear=False).to(device)

    # 损失函数选择逻辑：
    # - 若 NUM_CLASSES==1：用 BCE（输出 [N,1,H,W]）
    # - 否则：用 CE（输出 [N,C,H,W]，标签 [N,H,W]）
    criterion = torch.nn.BCEWithLogitsLoss() if NUM_CLASSES == 1 else torch.nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2)
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))

    best_f1 = -1.0
    save_path = "/kaggle/working/best_unet.pth"

    def batch_f1(pred, target, eps=1e-7):
        # pred/target 形状: [N,H,W]，值: 0/1
        pred = pred.float()
        target = target.float()
        tp = (pred * target).sum()
        fp = (pred * (1 - target)).sum()
        fn = ((1 - pred) * target).sum()
        p = tp / (tp + fp + eps)
        r = tp / (tp + fn + eps)
        f1 = 2 * p * r / (p + r + eps)
        return f1.item()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        total_f1 = 0.0
        n_batch = 0

        pbar = tqdm(dl, desc=f"Epoch {epoch}/{EPOCHS}")
        for imgs, masks in pbar:
            imgs = imgs.to(device)
            masks = masks.to(device)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=device.type, enabled=(USE_AMP and device.type == "cuda")):
                logits = model(imgs)
                if NUM_CLASSES == 1:
                    loss = criterion(logits.squeeze(1), masks.float())
                    pred = (torch.sigmoid(logits.squeeze(1)) > 0.5).long()
                else:
                    loss = criterion(logits, masks)
                    pred = torch.argmax(logits, dim=1)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            f1 = batch_f1(pred, (masks > 0).long())
            total_loss += loss.item()
            total_f1 += f1
            n_batch += 1
            pbar.set_postfix(loss=loss.item(), f1=f1)

        avg_loss = total_loss / max(n_batch, 1)
        avg_f1 = total_f1 / max(n_batch, 1)
        scheduler.step(avg_f1)

        print(f"[Epoch {epoch}] avg_loss={avg_loss:.4f}, avg_f1={avg_f1:.4f}, lr={optimizer.param_groups[0]['lr']:.2e}")

        # 保存最优模型
        if avg_f1 > best_f1:
            best_f1 = avg_f1
            state = model.state_dict()
            state["mask_values"] = [0, 255]  # 可选，和你的原项目兼容
            torch.save(state, save_path)
            print(f"  -> 保存新最优模型: {save_path}, best_f1={best_f1:.4f}")

    print("训练结束，最佳 F1:", best_f1)

## 4) Kaggle Notebook：推理、可视化与模型导出

本节完成：

1. 加载最佳权重
2. 对单图/批量做推理
3. 保存可视化结果与掩码
4. 可选导出 TorchScript / ONNX
5. 常见报错排查

In [ ]:
import os
import matplotlib.pyplot as plt

# 1) 加载最佳权重（如果上节训练得到）
infer_model = UNet(n_channels=3, n_classes=NUM_CLASSES, bilinear=False).to(device)
best_ckpt = "/kaggle/working/best_unet.pth"

if os.path.exists(best_ckpt):
    sd = torch.load(best_ckpt, map_location=device)
    sd.pop("mask_values", None)
    infer_model.load_state_dict(sd)
    print("已加载最佳模型:", best_ckpt)
else:
    print("未找到 best_unet.pth，请先运行训练或手动指定权重路径")

infer_model.eval()

# 2) 单图推理示例
if len(img_list) > 0:
    img = Image.open(img_list[0]).convert("RGB").resize((256, 256))
    arr = np.array(img).astype(np.float32) / 255.0
    x = torch.tensor(np.transpose(arr, (2, 0, 1)), dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = infer_model(x)
        if NUM_CLASSES == 1:
            pred = (torch.sigmoid(logits) > 0.5).long().squeeze().cpu().numpy()
        else:
            pred = torch.argmax(logits, dim=1).squeeze().cpu().numpy()

    # 保存掩码到 /kaggle/working
    out_mask_path = "/kaggle/working/pred_mask_0.png"
    Image.fromarray((pred * 255).astype(np.uint8)).save(out_mask_path)
    print("单图预测掩码已保存:", out_mask_path)

    # 可视化
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 3, 1)
    plt.title("Input")
    plt.imshow(arr)
    plt.axis("off")

    if len(mask_list) > 0:
        gt = np.array(Image.open(mask_list[0]).convert("L").resize((256, 256)))
        plt.subplot(1, 3, 2)
        plt.title("GT")
        plt.imshow(gt, cmap="gray")
        plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.title("Pred")
    plt.imshow(pred, cmap="gray")
    plt.axis("off")
    plt.show()

# 3) 批量推理示例（前10张）
os.makedirs("/kaggle/working/preds", exist_ok=True)
for i, p in enumerate(img_list[:10]):
    img = Image.open(p).convert("RGB").resize((256, 256))
    arr = np.array(img).astype(np.float32) / 255.0
    x = torch.tensor(np.transpose(arr, (2, 0, 1)), dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = infer_model(x)
        if NUM_CLASSES == 1:
            pred = (torch.sigmoid(logits) > 0.5).long().squeeze().cpu().numpy()
        else:
            pred = torch.argmax(logits, dim=1).squeeze().cpu().numpy()

    save_p = f"/kaggle/working/preds/pred_{i:03d}.png"
    Image.fromarray((pred * 255).astype(np.uint8)).save(save_p)

print("批量推理完成，输出目录: /kaggle/working/preds")

# 4) 可选导出 TorchScript（常用）
try:
    example = torch.randn(1, 3, 256, 256).to(device)
    traced = torch.jit.trace(infer_model, example)
    ts_path = "/kaggle/working/unet_traced.pt"
    traced.save(ts_path)
    print("TorchScript 导出成功:", ts_path)
except Exception as e:
    print("TorchScript 导出失败:", e)

# 5) 常见报错排查（文本输出）
print("\n常见问题排查：")
print("1) 尺寸不匹配：检查输入尺寸与网络下采样倍数（建议 H/W 可被 16 整除）")
print("2) 类别数不匹配：权重训练时的 n_classes 必须与推理时一致")
print("3) 掩码编码错误：确保训练和推理都使用同一套标签编码（如 0/1 或 0/255）")
print("4) OOM：减小 batch_size、输入尺寸，或启用 AMP")